In [1]:
import xarray as xr

ds = xr.open_zarr(r"D:\00. PHD Project\2nd Objective Causal S2S rainfall\DATA\ECMWF\s2s_reforecast.zarr\s2s_reforecast.zarr")

print(ds)


d:\ANACONDA\envs\pytorch-gpu\lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


<xarray.Dataset> Size: 7GB
Dimensions:                  (time: 3720, step: 43, lat: 33, lon: 35)
Coordinates:
  * lat                      (lat) float64 264B 38.5 37.5 36.5 ... 8.5 7.5 6.5
  * lon                      (lon) float64 280B 66.5 67.5 68.5 ... 99.5 100.5
  * step                     (step) int64 344B 0 1 2 3 4 5 ... 37 38 39 40 41 42
  * time                     (time) datetime64[ns] 30kB 2005-01-01 ... 2024-1...
Data variables:
    10m_u_component_of_wind  (time, step, lat, lon) float32 739MB dask.array<chunksize=(10, 22, 17, 35), meta=np.ndarray>
    10m_v_component_of_wind  (time, step, lat, lon) float32 739MB dask.array<chunksize=(10, 22, 17, 35), meta=np.ndarray>
    2m_temperature           (time, step, lat, lon) float32 739MB dask.array<chunksize=(10, 22, 17, 35), meta=np.ndarray>
    mean_sea_level_pressure  (time, step, lat, lon) float32 739MB dask.array<chunksize=(10, 22, 17, 35), meta=np.ndarray>
    specific_humidity_1000   (time, step, lat, lon) float32 739MB d

In [1]:
import xarray as xr

ds = xr.open_zarr(r"D:\00. PHD Project\2nd Objective Causal S2S rainfall\DATA\ECMWF_ALL_VARIABLES\s2s_reforecast.zarr\s2s_reforecast_sorted.zarr")

print(ds)


d:\ANACONDA\envs\pytorch-gpu\lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


<xarray.Dataset> Size: 16GB
Dimensions:                   (time: 3720, step: 43, lat: 33, lon: 35)
Coordinates:
  * lat                       (lat) float64 264B 38.5 37.5 36.5 ... 8.5 7.5 6.5
  * lon                       (lon) float64 280B 66.5 67.5 68.5 ... 99.5 100.5
  * step                      (step) int64 344B 0 1 2 3 4 5 ... 38 39 40 41 42
  * time                      (time) datetime64[ns] 30kB 2005-01-01 ... 2024-...
Data variables: (12/21)
    10m_u_component_of_wind   (time, step, lat, lon) float32 739MB dask.array<chunksize=(20, 43, 33, 35), meta=np.ndarray>
    10m_v_component_of_wind   (time, step, lat, lon) float32 739MB dask.array<chunksize=(20, 43, 33, 35), meta=np.ndarray>
    2m_temperature            (time, step, lat, lon) float32 739MB dask.array<chunksize=(20, 43, 33, 35), meta=np.ndarray>
    mean_sea_level_pressure   (time, step, lat, lon) float32 739MB dask.array<chunksize=(20, 43, 33, 35), meta=np.ndarray>
    specific_humidity_1000    (time, step, lat, lon) 

In [2]:
import xarray as xr

ds = xr.open_zarr(r"D:\00. PHD Project\DATA\IMD AB\IMD_rainfall_0p25.zarr\IMD_rainfall_0p25.zarr")

print(ds)

<xarray.Dataset> Size: 4GB
Dimensions:  (lat: 129, lon: 135, time: 31046)
Coordinates:
  * lat      (lat) float64 1kB 6.5 6.75 7.0 7.25 7.5 ... 37.75 38.0 38.25 38.5
  * lon      (lon) float64 1kB 66.5 66.75 67.0 67.25 ... 99.25 99.5 99.75 100.0
  * time     (time) datetime64[ns] 248kB 1941-01-01 1941-01-02 ... 2025-12-31
Data variables:
    rain     (time, lat, lon) float64 4GB dask.array<chunksize=(30, 129, 135), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.7
    comment:      
    crs:          epsg:4326
    history:      2026-06-19 06:45:25.930728 Python
    references:   
    source:       https://imdpune.gov.in/
    title:        IMD gridded data


## COMPREHENSIVE DATA ANALYSIS

In [3]:
"""
S2S DOWNSCALING PROJECT — DATASET AUDIT SCRIPT
================================================
Purpose : Thoroughly characterize the ECMWF IFS S2S reforecast (predictor, 1deg)
          and IMD gridded rainfall (predictand, 0.25deg) datasets BEFORE any
          modeling decisions are locked in.

Run this once per dataset refresh. It writes:
  - a text report (audit_report.txt)
  - diagnostic PNGs (grid alignment, missingness, distributions, climatology,
    accumulation check, lead-time bias-growth curve)
  - a small JSON of numeric summary stats you can paste into the paper's
    data-description section.

Edit the CONFIG block only. Nothing else should need touching for a first pass.
"""

import os
import json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# CONFIG — EDIT THESE PATHS / NAMES
# ============================================================
ECMWF_ZARR = r"D:\00. PHD Project\2nd Objective Causal S2S rainfall\DATA\ECMWF_ALL_VARIABLES\s2s_reforecast.zarr\s2s_reforecast_sorted.zarr"
IMD_ZARR   = r"D:\00. PHD Project\DATA\IMD AB\IMD_rainfall_0p25.zarr\IMD_rainfall_0p25.zarr"

OUT_DIR = r"./s2s_audit_output"

# Name of the precipitation variable inside the ECMWF zarr (check ds.data_vars first
# if unsure — this script will print all var names regardless).
ECMWF_PRECIP_VAR_GUESS = ["total_precipitation", "tp", "precipitation"]

IMD_PRECIP_VAR = "rain"

# For distribution / bias plots, restrict IMD's very long record (1941-2025) to the
# years that overlap the ECMWF reforecast period, to keep things comparable.
IMD_TIME_START = "2005-01-01"
IMD_TIME_END   = "2024-12-31"

os.makedirs(OUT_DIR, exist_ok=True)
report_lines = []


def log(msg):
    print(msg)
    report_lines.append(str(msg))


def section(title):
    log("\n" + "=" * 70)
    log(title)
    log("=" * 70)


# ============================================================
# 1. LOAD
# ============================================================
section("1. LOADING DATASETS")

ds_ec = xr.open_zarr(ECMWF_ZARR)
ds_imd = xr.open_zarr(IMD_ZARR)

log(f"ECMWF vars: {list(ds_ec.data_vars)}")
log(f"IMD vars:   {list(ds_imd.data_vars)}")

precip_var = None
for cand in ECMWF_PRECIP_VAR_GUESS:
    if cand in ds_ec.data_vars:
        precip_var = cand
        break
if precip_var is None:
    log("!! Could not auto-detect ECMWF precip variable from guess list. "
        "Set ECMWF_PRECIP_VAR_GUESS or edit precip_var manually. "
        "Skipping precip-specific checks.")
else:
    log(f"Detected ECMWF precip variable: '{precip_var}'")


# ============================================================
# 2. GRID / DOMAIN ALIGNMENT CHECK   (critical — flagged decision #1)
# ============================================================
section("2. GRID ALIGNMENT — ECMWF (1deg) vs IMD (0.25deg)")

ec_lat = ds_ec["lat"].values
ec_lon = ds_ec["lon"].values
imd_lat = ds_imd["lat"].values
imd_lon = ds_imd["lon"].values

log(f"ECMWF lat range: {ec_lat.min()} to {ec_lat.max()}  (n={len(ec_lat)}, "
    f"{'ascending' if ec_lat[0] < ec_lat[-1] else 'DESCENDING'})")
log(f"ECMWF lon range: {ec_lon.min()} to {ec_lon.max()}  (n={len(ec_lon)})")
log(f"IMD   lat range: {imd_lat.min()} to {imd_lat.max()}  (n={len(imd_lat)}, "
    f"{'ascending' if imd_lat[0] < imd_lat[-1] else 'DESCENDING'})")
log(f"IMD   lon range: {imd_lon.min()} to {imd_lon.max()}  (n={len(imd_lon)})")

if ec_lat[0] > ec_lat[-1]:
    log(">>> ACTION REQUIRED: ECMWF lat is descending, IMD lat is ascending. "
        "You must flip one before any xr.interp / regridding, or you'll get "
        "silently mirrored fields.")

lat_overlap = (max(ec_lat.min(), imd_lat.min()), min(ec_lat.max(), imd_lat.max()))
lon_overlap = (max(ec_lon.min(), imd_lon.min()), min(ec_lon.max(), imd_lon.max()))
log(f"Overlapping domain: lat {lat_overlap}, lon {lon_overlap}")
if lon_overlap[1] < lon_overlap[0] or lat_overlap[1] < lat_overlap[0]:
    log(">>> WARNING: no spatial overlap detected — check CRS/wrap-around (0-360 vs -180-180).")

# resolution ratio
ec_res = abs(ec_lat[1] - ec_lat[0])
imd_res = abs(imd_lat[1] - imd_lat[0])
log(f"ECMWF resolution: {ec_res} deg | IMD resolution: {imd_res} deg | "
    f"ratio: {ec_res/imd_res:.2f}x")

# Plot both grids overlaid
fig, ax = plt.subplots(figsize=(7, 6))
LON_EC, LAT_EC = np.meshgrid(ec_lon, ec_lat)
LON_IMD, LAT_IMD = np.meshgrid(imd_lon, imd_lat)
ax.scatter(LON_EC, LAT_EC, s=8, c="red", label=f"ECMWF ({ec_res}deg)", alpha=0.6)
ax.scatter(LON_IMD, LAT_IMD, s=1, c="blue", label=f"IMD ({imd_res}deg)", alpha=0.3)
ax.legend()
ax.set_title("Grid alignment check: ECMWF vs IMD")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
fig.savefig(os.path.join(OUT_DIR, "01_grid_alignment.png"), dpi=140, bbox_inches="tight")
plt.close(fig)
log("Saved 01_grid_alignment.png")


# ============================================================
# 3. TEMPORAL COVERAGE / INIT FREQUENCY  (flagged decision #5)
# ============================================================
section("3. ECMWF REFORECAST TEMPORAL STRUCTURE")

ec_time = pd.to_datetime(ds_ec["time"].values)
log(f"Number of init dates: {len(ec_time)}")
log(f"Range: {ec_time.min()} to {ec_time.max()}")

df_time = pd.DataFrame({"date": ec_time})
df_time["year"] = df_time.date.dt.year
df_time["month"] = df_time.date.dt.month
df_time["doy"] = df_time.date.dt.dayofyear
counts_per_year = df_time.groupby("year").size()
log("Init counts per year:\n" + counts_per_year.to_string())

# gap analysis
diffs = ec_time.to_series().diff().dropna().dt.days
log(f"Init date gaps (days): min={diffs.min()}, median={diffs.median()}, "
    f"max={diffs.max()}, mode={diffs.mode().iloc[0]}")
log("If gaps are NOT uniform (e.g. mix of 3/4 day gaps), the reforecast is issued "
    "on fixed calendar weekdays (typical ECMWF S2S: twice weekly). Confirm this "
    "matches ECMWF's documented IFS Cycle 50r1 hindcast schedule before assuming "
    "daily-equivalent sampling in train/val splits.")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df_time.date, np.ones(len(df_time)), "|", markersize=10)
ax.set_title("ECMWF reforecast initialization dates")
ax.set_yticks([])
fig.savefig(os.path.join(OUT_DIR, "02_init_dates_timeline.png"), dpi=140, bbox_inches="tight")
plt.close(fig)
log("Saved 02_init_dates_timeline.png")

# check IMD daily continuity in the overlap period
imd_sub = ds_imd.sel(time=slice(IMD_TIME_START, IMD_TIME_END))
imd_time = pd.to_datetime(imd_sub["time"].values)
full_range = pd.date_range(imd_time.min(), imd_time.max(), freq="D")
missing_days = full_range.difference(imd_time)
log(f"\nIMD (subset {IMD_TIME_START}-{IMD_TIME_END}): {len(imd_time)} days present, "
    f"{len(missing_days)} missing calendar days out of {len(full_range)} expected.")
if len(missing_days) > 0:
    log(f"First few missing IMD days: {missing_days[:10].tolist()}")


# ============================================================
# 4. ACCUMULATION CHECK for precipitation  (flagged decision #3)
# ============================================================
section("4. PRECIPITATION ACCUMULATION CHECK (ECMWF)")

if precip_var is not None:
    sample = ds_ec[precip_var].isel(time=0).load()  # (step, lat, lon)
    step_means = sample.mean(dim=["lat", "lon"]).values
    log("Domain-mean precip value by lead step (first init date):")
    for s, v in zip(ds_ec["step"].values, step_means):
        log(f"  step {s:>3d}: {v:.6f}")
    is_monotonic = np.all(np.diff(step_means) >= -1e-9)
    log(f"Monotonically non-decreasing across steps: {is_monotonic}")
    if is_monotonic and step_means[-1] > step_means[1] * 5:
        log(">>> LIKELY STILL CUMULATIVE (accumulated-from-init). You MUST apply "
            "`.diff(dim='step')` (with the first step handled/zero-filled, exactly "
            "as done for SSRD in your reference causality.py) before using this as "
            "an instantaneous/daily rainfall predictor. Verify against ECMWF S2S "
            "documentation for this parameter's step_type.")
    else:
        log("Looks like it may already be de-accumulated (per-step values), but "
            "verify against ECMWF param documentation — do not assume from one sample.")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(ds_ec["step"].values, step_means, marker="o")
    ax.set_xlabel("lead step (days)"); ax.set_ylabel("domain-mean precip value")
    ax.set_title(f"{precip_var}: domain mean vs lead step (single init)")
    fig.savefig(os.path.join(OUT_DIR, "03_precip_accumulation_check.png"), dpi=140, bbox_inches="tight")
    plt.close(fig)
    log("Saved 03_precip_accumulation_check.png")
else:
    log("Skipped — precip variable not auto-detected.")


# ============================================================
# 5. MISSING DATA / NaN AUDIT
# ============================================================
section("5. MISSING DATA AUDIT")

log("--- ECMWF ---")
nan_summary = {}
for var in ds_ec.data_vars:
    da = ds_ec[var]
    # sample-based NaN check (full dataset is large; use a representative subset)
    sample = da.isel(time=slice(0, 50)).load()
    n_nan = int(np.isnan(sample.values).sum())
    n_total = sample.size
    pct = 100 * n_nan / n_total
    nan_summary[var] = pct
    log(f"  {var:35s}: {pct:.3f}% NaN (sampled on first 50 inits)")

log("\n--- IMD ---")
imd_sample = ds_imd[IMD_PRECIP_VAR].isel(time=slice(0, 200)).load()
n_nan = int(np.isnan(imd_sample.values).sum())
n_total = imd_sample.size
log(f"  {IMD_PRECIP_VAR}: {100*n_nan/n_total:.3f}% NaN (sampled on first 200 days) "
    f"— NaNs here are typically ocean/out-of-India-landmass mask cells; confirm "
    f"NaN pattern matches a land mask, not data dropout.")

# visualize IMD NaN mask (spatial) — this doubles as a de-facto land mask check
fig, ax = plt.subplots(figsize=(6, 5))
mask_img = np.isnan(imd_sample.isel(time=0).values)
ax.imshow(mask_img, origin="lower", extent=[imd_lon.min(), imd_lon.max(), imd_lat.min(), imd_lat.max()])
ax.set_title("IMD NaN mask (single day) — should look like ocean/border mask")
fig.savefig(os.path.join(OUT_DIR, "04_imd_nan_mask.png"), dpi=140, bbox_inches="tight")
plt.close(fig)
log("Saved 04_imd_nan_mask.png")


# ============================================================
# 6. DISTRIBUTIONS  (zero-inflation, tails — matters a lot for precip loss functions)
# ============================================================
section("6. VARIABLE DISTRIBUTIONS")

log("--- IMD rainfall distribution (overlap period) ---")
imd_vals = imd_sub[IMD_PRECIP_VAR].values.ravel()
imd_vals = imd_vals[~np.isnan(imd_vals)]
zero_frac = np.mean(imd_vals <= 0.1)  # IMD convention: <0.1mm often treated as trace/dry
log(f"  n={len(imd_vals)}, dry-day fraction (<=0.1mm): {zero_frac:.3f}")
for p in [50, 90, 95, 99, 99.9]:
    log(f"  p{p}: {np.percentile(imd_vals, p):.2f} mm")
log(f"  max: {imd_vals.max():.2f} mm")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(imd_vals, bins=100)
ax[0].set_yscale("log")
ax[0].set_title("IMD rainfall histogram (log-y)")
ax[0].set_xlabel("mm/day")
ax[1].hist(np.log1p(imd_vals), bins=100)
ax[1].set_title("IMD log1p(rainfall) histogram")
fig.savefig(os.path.join(OUT_DIR, "05_imd_precip_distribution.png"), dpi=140, bbox_inches="tight")
plt.close(fig)
log("Saved 05_imd_precip_distribution.png")

if precip_var is not None:
    log("\n--- ECMWF precip distribution (short-lead steps only, first 100 inits, raw units) ---")
    ec_precip_sample = ds_ec[precip_var].isel(time=slice(0, 100), step=slice(0, 3)).load().values
    ec_precip_sample = ec_precip_sample[~np.isnan(ec_precip_sample)]
    for p in [50, 90, 95, 99, 99.9]:
        log(f"  p{p}: {np.percentile(ec_precip_sample, p):.6f} (raw units — confirm m vs mm)")
    log(f"  max: {ec_precip_sample.max():.6f}")
    log("  NOTE: ECMWF precip is often in meters, IMD in mm — a 1000x unit mismatch "
        "is a classic silent bug. Explicitly convert before comparison/training.")


# ============================================================
# 7. LEAD-TIME SKILL DEGRADATION PROXY (domain-mean bias growth)
#     — cheap sanity check before full verification; not a substitute for proper
#       CRPS/skill scoring later.
# ============================================================
section("7. LEAD-TIME BEHAVIOR (domain-mean, precip variable)")

if precip_var is not None:
    step_domain_means = ds_ec[precip_var].isel(time=slice(0, 100)).mean(
        dim=["time", "lat", "lon"], skipna=True).compute()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(ds_ec["step"].values, step_domain_means.values, marker="o")
    ax.set_xlabel("lead step (days)")
    ax.set_ylabel("domain & ensemble/init mean precip")
    ax.set_title("Precip evolution with lead time (sanity check, 100 inits)")
    fig.savefig(os.path.join(OUT_DIR, "06_leadtime_meanprecip.png"), dpi=140, bbox_inches="tight")
    plt.close(fig)
    log("Saved 06_leadtime_meanprecip.png — inspect for spin-up drift or "
        "unphysical jumps at specific lead steps (common at step 0 or step 1 "
        "boundary due to de-accumulation artifacts).")


# ============================================================
# 8. MEMORY / CHUNKING ASSESSMENT  (practical, for Claude-assisted training later)
# ============================================================
section("8. MEMORY & CHUNKING")

log("ECMWF dataset:")
log(f"  Total size (all vars): {ds_ec.nbytes / 1e9:.2f} GB")
for var in list(ds_ec.data_vars)[:3]:
    da = ds_ec[var]
    log(f"  {var}: shape={da.shape}, chunks={da.chunks}, dtype={da.dtype}")
log("IMD dataset:")
log(f"  Total size: {ds_imd.nbytes / 1e9:.2f} GB")
log(f"  {IMD_PRECIP_VAR}: shape={ds_imd[IMD_PRECIP_VAR].shape}, "
    f"chunks={ds_imd[IMD_PRECIP_VAR].chunks}, dtype={ds_imd[IMD_PRECIP_VAR].dtype}")
log("\nRecommendation: with 21 ECMWF variables x 739MB each = ~16GB total, plan "
    "for chunked/lazy loading (dask) throughout preprocessing — do NOT .load() the "
    "full multi-variable stack into memory at once when you get to Phase 3+ "
    "(pressure-level variables). Rechunk to (time=small, step=full, lat=full, lon=full) "
    "for per-sample tensor extraction during training.")


# ============================================================
# 9. CROSS-VARIABLE CORRELATION (for PCA / redundancy assessment — decision #8)
# ============================================================
section("9. CROSS-VARIABLE CORRELATION (redundancy check for PCA decision)")

sample_vars = [v for v in ds_ec.data_vars][:8]  # cap for speed; expand as needed
flat_data = {}
for v in sample_vars:
    arr = ds_ec[v].isel(time=slice(0, 30), step=0).load().values.ravel()
    flat_data[v] = arr

corr_df = pd.DataFrame(flat_data).corr()
log(corr_df.round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr_df.values, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(sample_vars))); ax.set_xticklabels(sample_vars, rotation=90)
ax.set_yticks(range(len(sample_vars))); ax.set_yticklabels(sample_vars)
plt.colorbar(im)
ax.set_title("Cross-variable correlation (step=0 sample)")
fig.savefig(os.path.join(OUT_DIR, "07_variable_correlation.png"), dpi=140, bbox_inches="tight")
plt.close(fig)
log("Saved 07_variable_correlation.png — variable pairs with |r|>0.9 are strong "
    "PCA/redundancy-reduction candidates (e.g. check the 4 specific-humidity "
    "levels and wind components across levels here).")


# ============================================================
# WRITE OUTPUTS
# ============================================================
with open(os.path.join(OUT_DIR, "audit_report.txt"), "w") as f:
    f.write("\n".join(report_lines))

summary = {
    "ecmwf_vars": list(ds_ec.data_vars),
    "ecmwf_n_init_dates": len(ec_time),
    "ecmwf_lead_steps": int(len(ds_ec["step"])),
    "ecmwf_resolution_deg": float(ec_res),
    "imd_resolution_deg": float(imd_res),
    "imd_dry_day_fraction": float(zero_frac),
    "nan_pct_by_ecmwf_var": nan_summary,
}
with open(os.path.join(OUT_DIR, "audit_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\nDone. See {OUT_DIR}/audit_report.txt, audit_summary.json, and PNGs.")


1. LOADING DATASETS
ECMWF vars: ['10m_u_component_of_wind', '10m_v_component_of_wind', '2m_temperature', 'mean_sea_level_pressure', 'specific_humidity_1000', 'specific_humidity_200', 'specific_humidity_500', 'specific_humidity_850', 'temperature_1000', 'temperature_200', 'temperature_500', 'temperature_850', 'total_precipitation', 'u_component_of_wind_1000', 'u_component_of_wind_200', 'u_component_of_wind_500', 'u_component_of_wind_850', 'v_component_of_wind_1000', 'v_component_of_wind_200', 'v_component_of_wind_500', 'v_component_of_wind_850']
IMD vars:   ['rain']
Detected ECMWF precip variable: 'total_precipitation'

2. GRID ALIGNMENT — ECMWF (1deg) vs IMD (0.25deg)
ECMWF lat range: 6.5 to 38.5  (n=33, DESCENDING)
ECMWF lon range: 66.5 to 100.5  (n=35)
IMD   lat range: 6.5 to 38.5  (n=129, ascending)
IMD   lon range: 66.5 to 100.0  (n=135)
>>> ACTION REQUIRED: ECMWF lat is descending, IMD lat is ascending. You must flip one before any xr.interp / regridding, or you'll get silently m

C:\Users\USMAN\AppData\Local\Temp\ipykernel_60200\660451514.py:265: RuntimeWarning: invalid value encountered in log1p
  ax[1].hist(np.log1p(imd_vals), bins=100)


Saved 05_imd_precip_distribution.png

--- ECMWF precip distribution (short-lead steps only, first 100 inits, raw units) ---
  p50: 0.000000 (raw units — confirm m vs mm)
  p90: 5.289597 (raw units — confirm m vs mm)
  p95: 10.814453 (raw units — confirm m vs mm)
  p99: 26.445312 (raw units — confirm m vs mm)
  p99.9: 54.894531 (raw units — confirm m vs mm)
  max: 241.816406
  NOTE: ECMWF precip is often in meters, IMD in mm — a 1000x unit mismatch is a classic silent bug. Explicitly convert before comparison/training.

7. LEAD-TIME BEHAVIOR (domain-mean, precip variable)
Saved 06_leadtime_meanprecip.png — inspect for spin-up drift or unphysical jumps at specific lead steps (common at step 0 or step 1 boundary due to de-accumulation artifacts).

8. MEMORY & CHUNKING
ECMWF dataset:
  Total size (all vars): 15.52 GB
  10m_u_component_of_wind: shape=(3720, 43, 33, 35), chunks=((20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 2